# Practice 37 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — tips: spot the bug

Each snippet below has exactly one bug. Find it, fix it, and explain in a comment what was wrong.

```python
tips = sns.load_dataset('tips')

# A — supposed to flag rows where tip > 20% of total_bill
tips['good_tipper'] = tips['tip'] / tips['total_bill'] > 20

# B — supposed to label size >= 5 as 'large', 3-4 as 'medium', else 'small'
conditions = [tips['size'] >= 3, tips['size'] >= 5]
choices = ['medium', 'large']
tips['size_label'] = np.select(conditions, choices, default='small')

# C — supposed to add boolean column: True where both total_bill > 20 and tip > 3
tips['both_high'] = tips['total_bill'] > 20 and tips['tip'] > 3

# D — supposed to get the mean tip for each day, sorted highest to lowest
tips.groupby('day')['tip'].mean().sort_values(ascending=True)
```

In [4]:
tips = sns.load_dataset('tips')

# Your fixes here

#A 
tips['good_tipper'] = (tips['tip']/tips['total_bill']) > 0.2

#B
conditions = [tips['size']>=5, tips['size']>=3]
choices = ['larger', 'medium']
tips['size_label']= np.select(conditions, choices, default = 'small')

#C
tips['both_high'] = (tips['total_bill']> 20) & (tips['tip']>3)

#D
tips.groupby('day')['tip'].mean().sort_values(ascending = False)


/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_34793/3979520361.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby('day')['tip'].mean().sort_values(ascending = False)


day
Sun     3.255132
Sat     2.993103
Thur    2.771452
Fri     2.734737
Name: tip, dtype: float64

---
## Level 2 — titanic: vectorized feature engineering

Load `sns.load_dataset('titanic')`. No `.apply()` anywhere.

Build these columns, all vectorized:

1. `family_size` — total people travelling together (sibsp + parch + 1 for self)
2. `fare_per_person` — fare divided by family_size
3. `travel_type` — use `np.select()`: `'solo'` (family_size == 1), `'small group'` (2–3), `'large group'` (4+)
4. `age_bracket` — `'child'` (age < 13), `'teen'` (13–17), `'adult'` (18–59), `'senior'` (60+). Handle NaN age: rows with NaN age should get `'unknown'` — make `default='unknown'` and write conditions so NaN naturally falls through.
5. Convert `travel_type` and `age_bracket` to `category`.

Then answer: which `travel_type` has the highest survival rate? Which `age_bracket`? Use groupby, no `.apply()`.

In [13]:
# Your code here


titanic = sns.load_dataset('titanic')

titanic['family_size'] = titanic['sibsp'] + titanic['parch'] +1
titanic['fare_per_person'] = titanic['fare']/titanic['family_size']
conditions = [titanic['family_size']>=4, titanic['family_size']>=2]
choices = ['large group','small_group']
titanic['travel_type'] =np.select(conditions, choices, default = 'solo')
conds =[titanic['age']>=60, titanic['age']>=18, titanic['age']>=13, titanic['age']>=0]
chois = ['senior','adult','teem','child']
titanic['age_bracket'] = np.select(conds, chois, default= 'unknown')

titanic[['travel_type','age_bracket']] = titanic[['travel_type','age_bracket']].astype('category')

titanic.groupby('travel_type', observed = True)['survived'].mean().idxmax()
titanic.groupby('age_bracket', observed = True)['survived'].mean().idxmax()




'child'

---
## Level 3 — penguins: five questions

Load `sns.load_dataset('penguins')`. Drop nulls. No `.apply()`.

1. What fraction of Gentoo penguins have both `bill_length_mm` above the **overall** median and `body_mass_g` above the **overall** median? One expression.
2. Add `size_score` = `(body_mass_g / body_mass_g.max()) + (flipper_length_mm / flipper_length_mm.max())` — fully vectorized. Which island has the highest mean `size_score`?
3. For each species, what is the correlation (`np.corrcoef`) between `bill_length_mm` and `bill_depth_mm`? Loop over species, print one line per species.
4. Add `bill_class` using `np.select()`: `'wide'` (bill_depth_mm > 18), `'narrow'` (bill_depth_mm < 15), `'mid'` otherwise. Build a plain Python list of species names (no duplicates) that appear in the `'wide'` class — use a list comprehension with `zip`.
5. Use `transform` to add each penguin's species-level mean `body_mass_g`. Then add `mass_diff` — how far each penguin is from its species mean. Which species has the largest spread (std of `mass_diff`)?

In [ ]:
# Your code here

penguins = sns.load_dataset('penguins').dropna()

((penguins[penguins['species']=='Gentoo']['bill_length_mm']>penguins['bill_length_mm'].median())&(penguins[penguins['species']=='Gentoo']['body_mass_g']>penguins['body_mass_g'].median())).mean()

penguins['size_score'] = penguins['body_mass_g']/penguins['body_mass_g'].max() + penguins['flipper_length_mm']/penguins['flipper_length_mm'].max()

penguins.groupby("island")['size_score'].mean().idxmax()

for s in penguins['species'].unique():
    ps = penguins.loc[penguins['species']==s,['bill_length_mm','bill_depth_mm']].dropna()
    print(s, np.corrcoef(ps['bill_length_mm'], ps['bill_depth_mm'])[0,1])

conds =[penguins['bill_depth_mm']>18, penguins['bill_depth_mm']<15]
chois = ['wide','narrow']
penguins['bill_class'] = np.select(conds, chois, default = 'mid')

set([species for species, bill_class in zip(penguins['species'], penguins['bill_class']) if bill_class == 'wide' ])

### what is a better way to do the above 

#penguins['sm'] = penguins.groupby('species')['body_mass_g'].transform('mean')#\
#penguins['mass_diff'] = penguins['body_mass_g'] - penguins['sm']
#penguins.groupby('species')['mass_diff'].std().idxmax()

Adelie 0.3858132004955791
Chinstrap 0.6535362081800429
Gentoo 0.6540233142726541


{'Adelie', 'Chinstrap'}

In [70]:
{species for species, bill_class in zip(penguins['species'], penguins['bill_class']) if bill_class == 'wide' }


{'Adelie', 'Chinstrap'}

---
## Level 4 — full pipeline

You have a student dataset below. Write a `.pipe()` chain — no `.apply()` anywhere.

- **`enrich(df)`** — add `total` (math + reading + writing), `avg_score` (total / 3), and `passed` (avg_score >= 60, boolean). Convert `gender` and `lunch` to `category`.
- **`classify(df)`** — add `grade` with `np.select()`: `'A'` (avg_score >= 90), `'B'` (>= 75), `'C'` (>= 60), `'F'` otherwise.
- **`flag(df)`** — add `needs_support`: True where `passed == False` or any individual subject score < 50 (math, reading, or writing).

After the chain:
- Pivot table: count of students by `gender` × `grade`, `observed=True`.
- What fraction of students with `lunch == 'free/reduced'` received a grade of `'F'`? Use `.loc` and `np.mean()`.
- Collect names of students who `needs_support` into a Python list using a list comprehension with `zip`.

In [69]:
students = pd.DataFrame({
    'name':    ['Alice','Bob','Carol','David','Eva','Frank','Grace','Hiro','Ivy','Jack',
                'Karen','Leo','Mia','Noah','Olivia','Pete','Quinn','Rosa','Sam','Tina'],
    'gender':  ['F','M','F','M','F','M','F','M','F','M',
                'F','M','F','M','F','M','F','F','M','F'],
    'lunch':   ['standard','free/reduced','standard','free/reduced','standard',
                'free/reduced','standard','standard','free/reduced','standard',
                'free/reduced','standard','free/reduced','standard','standard',
                'free/reduced','standard','free/reduced','standard','free/reduced'],
    'math':    [88, 45, 72, 55, 95, 38, 81, 63, 49, 77, 90, 58, 42, 85, 70, 33, 91, 66, 78, 52],
    'reading': [82, 50, 68, 60, 93, 44, 79, 71, 55, 80, 88, 62, 48, 83, 74, 40, 89, 72, 75, 58],
    'writing': [85, 48, 70, 57, 91, 41, 83, 67, 51, 78, 87, 60, 45, 81, 72, 37, 92, 69, 76, 55]
})
# Your code here

def enrich(df):
    df['total'] = df['math']+df['reading'] + df['writing']
    df['avg_score'] = df['total']/3
    df['passed'] = df['avg_score']>=60
    df[['gender','lunch']] = df[['gender','lunch']].astype('category')
    return df

def classify(df):
    cods= [df['avg_score']>=90, df['avg_score']>=75, df['avg_score']>=60]
    chs = ['A','B','C']
    df['grade'] = np.select(cods, chs, default='F')
    return df

def flag(df):
    mins = df[['math', 'reading','writing']].min(axis = 1)
    df['needs_support'] = ( df['passed'] == False) | (mins <50)

    #df['needs_support'] = mins <50

    return df



res = students.copy().pipe(enrich).pipe(classify).pipe(flag)
p = res.pivot_table(
    index = 'gender',
    columns = 'grade',
    values = 'name',
    aggfunc = len,
    fill_value = 0,
    observed = True
)
p


np.mean(res.loc[res['lunch'] == 'free/reduced']['grade'] == 'F')

# Collect names of students who `needs_support` into a Python list using a list comprehension with `zip`.

[name for name, n in zip(res['name'],res['needs_support']) if n]


['Bob', 'David', 'Frank', 'Ivy', 'Mia', 'Pete', 'Tina']